# Exploration scratchpad

Use this notebook to poke at the planner / search / writer / verifier in isolation
before promoting changes into `agent/*.py` and `prompts/*.txt`. Runs fully offline
by default (`mock=True`) so you can iterate without burning API credits.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))

from agent.llm import get_llm
from agent.planner import Planner
from agent.researcher import Researcher
from agent.writer import Writer
from agent.verifier import Verifier

llm = get_llm(mock=True)
planner, researcher, writer, verifier = Planner(llm), Researcher(), Writer(llm), Verifier(llm)

## 1. Inspect the planner's decomposition for a query

In [ ]:
plan = planner.plan("What are the trade-offs between RAG and long-context LLMs?", max_sub_questions=4)
print("Clarified goal:", plan.clarified_goal)
for sq in plan.sub_questions:
    print("-", sq.question, "->", sq.search_queries)

## 2. Gather sources and eyeball what actually came back

In [ ]:
sources = researcher.gather(plan)
print(f"{len(sources)} unique sources")
for s in sources[:5]:
    print(s.id, s.domain, s.title, "| fetched_ok=", s.fetched_ok, "| content_len=", len(s.content))

## 3. Draft, then verify -- this is the loop worth iterating on

In [ ]:
draft = writer.write(plan, sources)
print(draft.markdown)

In [ ]:
verification = verifier.verify(draft, sources)
print(f"faithfulness = {verification.faithfulness_score:.0%}")
for c in verification.claims:
    print(f"[{c.status.value:>20}] {c.claim}")

## 4. If you tweak a prompt, re-run from cell 1

Edit the relevant file under `prompts/` and re-run -- prompts are read fresh on every
`Planner()` / `Writer()` / `Verifier()` construction, no kernel restart needed if you
re-run the import cell.